In [11]:
# ==============================================================================
# Dual-Methodology Stance/Sentiment Pipeline
# 
# Group 1 (No Choice Key): Document-level VADER Sentiment Analysis
# Group 2 (Choice Key Populated): Sentence-level Regex Choice Filtering + ModernBERT
#
# Target MongoDB Collection: consensus.python_source_map_comments
# Metadata Collection:       consensus.python_source_map
# ==============================================================================

# ── CELL 1: SETUP, IMPORTS & ENVIRONMENT ──────────────────────────────────────
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import json, re, math, datetime
import numpy as np
import pandas as pd
import torch
from pymongo import MongoClient, UpdateOne
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from IPython.display import display

# Ensure VADER lexicon is downloaded
nltk.download("vader_lexicon", quiet=True)
vader_analyzer = SentimentIntensityAnalyzer()

# Execution Device
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using compute device: {device}")

Using compute device: mps


In [ ]:
# ── CELL 2: MONGO DB CONNECTION & RELATIONAL JOIN ──────────────────────────────
with open("config.json") as f:
    config = json.load(f)

client = MongoClient(config["mongo_uri"])
db = client["consensus"]
comments_coll = db["python_source_map_comments"]
map_coll = db["python_source_map"]

comments_df = pd.DataFrame(list(comments_coll.find({})))
map_df = pd.DataFrame(list(map_coll.find({})))

# Clean Choice field in source map
#if "Choice" in map_df.columns:
#    map_df["clean_choice"] = map_df["Choice"].fillna("").astype(str).str.strip()
#else:
#    map_df["clean_choice"] = ""

# Clean Choice field in source map (populated only when Target == 'Y')
if "Target" in map_df.columns and "Choice" in map_df.columns:
    map_df["clean_choice"] = np.where(
        map_df["Target"].fillna("N").astype(str).str.strip().str.upper() == "Y",
        map_df["Choice"].fillna("").astype(str).str.strip(),
        ""
    )
else:
    map_df["clean_choice"] = ""

map_clean = map_df[["Episode #", "clean_choice"]].drop_duplicates(subset=["Episode #"])

# Relational Join
if "Episode #" in comments_df.columns:
    df = comments_df.merge(map_clean, on="Episode #", how="left")
else:
    df = comments_df.copy()
    df["clean_choice"] = ""

df["clean_text"] = df["text"].fillna("").astype(str).str.strip()
df["clean_choice"] = df["clean_choice"].fillna("")

print(f"Loaded {len(df)} total comments across {df['Episode #'].nunique() if 'Episode #' in df else 0} episodes.")

KeyboardInterrupt: 

In [ ]:
# ── CELL 3: CHOICE REGEX BUILDER & SENTENCE SPLITTER ──────────────────────────
def build_choice_regex(choice_str):
    """
    Constructs a flexible regular expression for choice targets.
    Handles variations like PEP016, PEP-016, PEP 016, Option A vs Option-A.
    """
    if not choice_str or choice_str.lower() in ["nan", "none", ""]:
        return None
    
    choice_clean = choice_str.strip()
    
    # Standardize PEP numbering variations (e.g., PEP 16, PEP016, PEP-016)
    pep_match = re.match(r'^(pep)[-_\s]?0*(\d+)$', choice_clean, re.IGNORECASE)
    if pep_match:
        prefix, num = pep_match.groups()
        pattern = rf'\b{prefix}[-_\s]?0*{num}\b'
        return re.compile(pattern, re.IGNORECASE)
    
    # Generic word boundary regex with flexible spacing and hyphens
    escaped = re.escape(choice_clean)
    pattern = re.sub(r'\\\s+|\\-', r'[-_\\s]+', escaped)
    return re.compile(rf'\b{pattern}\b', re.IGNORECASE)

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', str(text))
    return [s.strip() for s in sentences if len(s.strip()) > 2]

In [ ]:
# ── CELL 4: LOAD MODERNBERT MODEL ─────────────────────────────────────────────
MODERNBERT_MODEL_ID = "answerdotai/ModernBERT-base"

print(f"Loading ModernBERT tokenizer and model: '{MODERNBERT_MODEL_ID}'...")
mb_tokenizer = AutoTokenizer.from_pretrained(MODERNBERT_MODEL_ID)
mb_model = AutoModelForSequenceClassification.from_pretrained(
    MODERNBERT_MODEL_ID, 
    num_labels=3
).to(device)
mb_model.eval()

print("ModernBERT successfully loaded.")

Loading ModernBERT tokenizer and model: 'answerdotai/ModernBERT-base'...


W1020 13:00:47.137000 41406 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ModernBERT successfully loaded.


In [ ]:
# ── CELL 5: SENTIMENT & STANCE PROCESSING FUNCTIONS ───────────────────────────
def analyze_vader_document(text):
    """
    Applies VADER sentiment analysis to full document text.
    Returns valence (-1, 0, 1) and metadata dictionary.
    """
    scores = vader_analyzer.polarity_scores(text)
    compound = scores["compound"]
    
    if compound >= 0.05:
        valence = 1
    elif compound <= -0.05:
        valence = -1
    else:
        valence = 0
        
    meta = {
        "method": "vader_document_level",
        "compound": float(compound),
        "pos": float(scores["pos"]),
        "neg": float(scores["neg"]),
        "neu": float(scores["neu"]),
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat()
    }
    return valence, meta

def analyze_modernbert_sentences(sentences, choice_str):
    """
    Runs ModernBERT targeted sequence classification on choice-filtered sentences.
    Returns valence (-1, 0, 1) and metadata dictionary.
    """
    combined_text = " ".join(sentences)
    inputs = mb_tokenizer(
        combined_text, 
        truncation=True, 
        max_length=8192, 
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        logits = mb_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        
    class_idx = int(np.argmax(probs))
    idx_to_valence = {0: -1, 1: 0, 2: 1}
    valence = idx_to_valence.get(class_idx, 0)
    
    meta = {
        "method": "modernbert_choice_targeted",
        "choice_target": choice_str,
        "matched_sentence_count": len(sentences),
        "matched_text_snippet": combined_text[:200],
        "probs": {
            "against": float(probs[0]),
            "neutral": float(probs[1]),
            "favor": float(probs[2])
        },
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat()
    }
    return valence, meta

In [ ]:
# ── CELL 6: EXECUTION LOOP & MONGO DB WRITEBACK ───────────────────────────────
bulk_ops = []
results_summary = []

for idx, row in df.iterrows():
    cid = row["_id"]
    text = row["clean_text"]
    choice_val = row["clean_choice"]
    choice_regex = build_choice_regex(choice_val)
    
    if choice_regex is None:
        # GROUP 1: No Choice populated -> Full Document VADER
        valence, meta = analyze_vader_document(text)
        group_type = "Group 1 (VADER Document)"
    else:
        # GROUP 2: Choice populated -> Filter matching sentences
        sentences = split_sentences(text)
        matching_sentences = [s for s in sentences if choice_regex.search(s)]
        
        if matching_sentences:
            valence, meta = analyze_modernbert_sentences(matching_sentences, choice_val)
            group_type = "Group 2 (ModernBERT Choice Sentence)"
        else:
            # Fallback to VADER if no sentences mention the choice keyword directly
            valence, meta = analyze_vader_document(text)
            meta["method"] = "vader_document_fallback_no_choice_sentences"
            meta["choice_target"] = choice_val
            group_type = "Group 2 (VADER Fallback - No Choice Match)"

    results_summary.append({
        "comment_id": str(cid),
        "Episode #": row.get("Episode #"),
        "choice": choice_val,
        "group_type": group_type,
        "valence": valence,
        "method": meta["method"]
    })

    # Prepare Mongo update doc
    bulk_ops.append(
        UpdateOne(
            {"_id": cid},
            {"$set": {
                "valence": valence,
                "valence_meta": meta
            }}
        )
    )

# Bulk Write to MongoDB
if bulk_ops:
    res = comments_coll.bulk_write(bulk_ops)
    print(f"MongoDB Update Complete: {res.modified_count} documents updated with 'valence' and 'valence_meta'.")

MongoDB Update Complete: 4068 documents updated with 'valence' and 'valence_meta'.


In [ ]:
# ── CELL 7: PIPELINE RESULTS SUMMARY ──────────────────────────────────────────
summary_df = pd.DataFrame(results_summary)

print("=" * 65)
print("          DUAL-METHODOLOGY PIPELINE SUMMARY FINDINGS            ")
print("=" * 65)
print(f"Total Comments Processed: {len(summary_df)}")
print("-" * 65)

print("\nProcessing Route Breakdown:")
print(summary_df["group_type"].value_counts().to_string())

print("\nOverall Valence Distribution:")
val_map = {1: "Positive / Favor (+1)", 0: "Neutral (0)", -1: "Negative / Against (-1)"}
for val, label in val_map.items():
    cnt = (summary_df["valence"] == val).sum()
    pct = (cnt / len(summary_df)) * 100 if len(summary_df) > 0 else 0
    print(f"  - {label:<25}: {cnt:5d} ({pct:5.1f}%)")

print("-" * 65)
print("Valence Crosstabulation by Processing Route:")
display(pd.crosstab(summary_df["group_type"], summary_df["valence"], margins=True))

          DUAL-METHODOLOGY PIPELINE SUMMARY FINDINGS            
Total Comments Processed: 4068
-----------------------------------------------------------------

Processing Route Breakdown:
group_type
Group 2 (VADER Fallback - No Choice Match)    2406
Group 1 (VADER Document)                      1344
Group 2 (ModernBERT Choice Sentence)           318

Overall Valence Distribution:
  - Positive / Favor (+1)    :  2923 ( 71.9%)
  - Neutral (0)              :   402 (  9.9%)
  - Negative / Against (-1)  :   743 ( 18.3%)
-----------------------------------------------------------------
Valence Crosstabulation by Processing Route:


valence,-1,0,1,All
group_type,,,,
Group 1 (VADER Document),285,118,941,1344
Group 2 (ModernBERT Choice Sentence),132,0,186,318
Group 2 (VADER Fallback - No Choice Match),326,284,1796,2406
All,743,402,2923,4068


In [ ]:
# ── CELL 8: EPISODE-LEVEL VALENCE & CONFIDENCE BREAKDOWN ──────────────────────
import pandas as pd
import numpy as np
from IPython.display import display

# Fetch stored valence, group_type, and valence_meta directly from MongoDB
enriched_records = []
cursor = comments_coll.find({}, {"_id": 1, "Episode #": 1, "valence": 1, "valence_meta": 1})

for doc in cursor:
    cid = str(doc["_id"])
    ep = doc.get("Episode #")
    val = doc.get("valence")
    meta = doc.get("valence_meta", {})
    
    method = meta.get("method", "unknown")
    
    # Calculate confidence score based on analysis method
    if "modernbert" in method:
        probs = meta.get("probs", {})
        conf = max(probs.values()) if probs else np.nan
        group = "Group 2 (ModernBERT Choice Sentence)"
    elif "vader" in method:
        # VADER confidence: magnitude of compound polarity score |compound|
        conf = abs(float(meta.get("compound", 0.0)))
        if "fallback" in method:
            group = "Group 2 (VADER Fallback - No Choice Match)"
        else:
            group = "Group 1 (VADER Document)"
    else:
        conf = np.nan
        group = "Unknown"
        
    enriched_records.append({
        "comment_id": cid,
        "Episode #": ep,
        "group_type": group,
        "valence": val,
        "method": method,
        "confidence": conf
    })

rep_df = pd.DataFrame(enriched_records)

print("=" * 75)
print("            EPISODE VALENCE & CONFIDENCE SUMMARY BREAKDOWN              ")
print("=" * 75)

# 1. Summary of Confidence Levels by Group Type / Method
print("\n--- 1. CONFIDENCE LEVEL STATISTICS BY METHODOLOGY GROUP ---")
conf_summary = rep_df.groupby("group_type")["confidence"].agg(
    Count="count",
    Mean="mean",
    Std="std",
    Min="min",
    Max="max"
).reset_index()

# Format statistics
conf_summary["Mean"] = conf_summary["Mean"].map("{:.4f}".format)
conf_summary["Std"] = conf_summary["Std"].map(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")
conf_summary["Min"] = conf_summary["Min"].map("{:.4f}".format)
conf_summary["Max"] = conf_summary["Max"].map("{:.4f}".format)
display(conf_summary)

# 2. Valence Counts Crosstabulated by Episode # and Group Type
print("\n--- 2. VALENCE BREAKDOWN BY DECISION EPISODE AND GROUP TYPE ---")
if "Episode #" in rep_df.columns and rep_df["Episode #"].notna().any():
    ep_group_valence = pd.crosstab(
        index=[rep_df["Episode #"], rep_df["group_type"]],
        columns=rep_df["valence"],
        margins=True,
        margins_name="Total"
    )
    
    # Rename valence columns (-1, 0, 1) for readability
    rename_map = {-1: "Against (-1)", 0: "Neutral (0)", 1: "Favor (+1)"}
    ep_group_valence.rename(columns=rename_map, inplace=True)
    
    # Display top 25 episode-group entries
    display(ep_group_valence.head(25))

    # 3. Consolidated Per-Episode Summary Table
    print("\n--- 3. CONSOLIDATED EPISODE SUMMARY (TOP 15 EPISODES) ---")
    ep_agg = rep_df.groupby("Episode #").agg(
        Total_Comments=("valence", "count"),
        Mean_Valence=("valence", "mean"),
        Mean_Confidence=("confidence", "mean"),
        Favor_Pct=("valence", lambda x: (x == 1).mean() * 100),
        Neutral_Pct=("valence", lambda x: (x == 0).mean() * 100),
        Against_Pct=("valence", lambda x: (x == -1).mean() * 100)
    ).reset_index()

    # Format percentages and means
    ep_agg["Mean_Valence"] = ep_agg["Mean_Valence"].map("{:+.3f}".format)
    ep_agg["Mean_Confidence"] = ep_agg["Mean_Confidence"].map("{:.4f}".format)
    ep_agg["Favor_Pct"] = ep_agg["Favor_Pct"].map("{:.1f}%".format)
    ep_agg["Neutral_Pct"] = ep_agg["Neutral_Pct"].map("{:.1f}%".format)
    ep_agg["Against_Pct"] = ep_agg["Against_Pct"].map("{:.1f}%".format)

    display(ep_agg.head(15))
else:
    print("No valid 'Episode #' field found in comments metadata.")

            EPISODE VALENCE & CONFIDENCE SUMMARY BREAKDOWN              

--- 1. CONFIDENCE LEVEL STATISTICS BY METHODOLOGY GROUP ---


,group_type,Count,Mean,Std,Min,Max
0,Group 1 (VADER Document),1344,0.5723,0.3192,0.0000,0.9988
1,Group 2 (ModernBERT Choice Sentence),318,0.5031,0.1044,0.3633,0.8061
2,Group 2 (VADER Fallback - No Choice Match),2406,0.5885,0.3342,0.0000,0.9998



--- 2. VALENCE BREAKDOWN BY DECISION EPISODE AND GROUP TYPE ---


,valence,Against (-1),Neutral (0),Favor (+1),Total
Episode #,group_type,,,,
1,Group 1 (VADER Document),3,1,14,18
2,Group 1 (VADER Document),2,1,6,9
3,Group 1 (VADER Document),0,1,3,4
4,Group 1 (VADER Document),1,0,4,5
5,Group 1 (VADER Document),3,1,10,14
6,Group 1 (VADER Document),17,7,77,101
7,Group 1 (VADER Document),10,1,20,31
8,Group 1 (VADER Document),9,2,32,43
9,Group 1 (VADER Document),1,0,6,7



--- 3. CONSOLIDATED EPISODE SUMMARY (TOP 15 EPISODES) ---


,Episode #,Total_Comments,Mean_Valence,Mean_Confidence,Favor_Pct,Neutral_Pct,Against_Pct
0,1,18,+0.611,0.6716,77.8%,5.6%,16.7%
1,2,9,+0.444,0.4042,66.7%,11.1%,22.2%
2,3,4,+0.750,0.3337,75.0%,25.0%,0.0%
3,4,5,+0.600,0.6475,80.0%,0.0%,20.0%
4,5,14,+0.500,0.4842,71.4%,7.1%,21.4%
5,6,101,+0.594,0.6708,76.2%,6.9%,16.8%
6,7,31,+0.323,0.5862,64.5%,3.2%,32.3%
7,8,43,+0.535,0.5391,74.4%,4.7%,20.9%
8,9,7,+0.714,0.7295,85.7%,0.0%,14.3%
9,10,40,+0.475,0.5127,67.5%,12.5%,20.0%


In [13]:
# ── CELL 9: COMMENT SAMPLES AT MINIMUM, NEAR-MEAN, AND MAXIMUM CONFIDENCE ──────
import pandas as pd
import numpy as np
from IPython.display import display

# 1. Fetch evaluated comments with text and valence metadata from MongoDB
cursor = comments_coll.find(
    {"valence": {"$exists": True}},
    {"_id": 1, "Episode #": 1, "text": 1, "valence": 1, "valence_meta": 1}
)

records = []
for doc in cursor:
    cid = str(doc["_id"])
    ep = doc.get("Episode #")
    text = str(doc.get("text", "")).strip().replace("\n", " ")
    val = doc.get("valence")
    meta = doc.get("valence_meta", {})
    method = meta.get("method", "unknown")

    # Extract confidence according to methodology
    if "modernbert" in method:
        probs = meta.get("probs", {})
        conf = max(probs.values()) if probs else np.nan
        method_label = "ModernBERT (Choice)"
    elif "vader" in method:
        conf = abs(float(meta.get("compound", 0.0)))
        method_label = "VADER (Document)" if "fallback" not in method else "VADER (Fallback)"
    else:
        conf = np.nan
        method_label = "Unknown"

    if pd.notna(ep) and pd.notna(conf) and text:
        records.append({
            "comment_id": cid,
            "Episode #": ep,
            "text": text,
            "valence": val,
            "method": method_label,
            "confidence": conf
        })

df_conf = pd.DataFrame(records)

# 2. Select 4 representative decision episodes with at least 5 comments
sampled_episodes = (
    df_conf.groupby("Episode #")
    .filter(lambda x: len(x) >= 5)["Episode #"]
    .unique()[:4]
)

sampled_rows = []

for ep in sampled_episodes:
    ep_df = df_conf[df_conf["Episode #"] == ep].copy()
    ep_mean_conf = ep_df["confidence"].mean()

    # Min Confidence Comment
    min_idx = ep_df["confidence"].idxmin()
    min_row = ep_df.loc[min_idx].to_dict()
    min_row["Confidence_Tier"] = "Min Confidence"

    # Near-Mean Confidence Comment (closest distance to episode mean)
    ep_df["dist_from_mean"] = (ep_df["confidence"] - ep_mean_conf).abs()
    mean_idx = ep_df["dist_from_mean"].idxmin()
    mean_row = ep_df.loc[mean_idx].to_dict()
    mean_row["Confidence_Tier"] = f"Near-Mean ({ep_mean_conf:.3f})"

    # Max Confidence Comment
    max_idx = ep_df["confidence"].idxmax()
    max_row = ep_df.loc[max_idx].to_dict()
    max_row["Confidence_Tier"] = "Max Confidence"

    for r in [min_row, mean_row, max_row]:
        r["text_snippet"] = r["text"][:150] + ("..." if len(r["text"]) > 150 else "")
        sampled_rows.append(r)

samples_df = pd.DataFrame(sampled_rows)

print("=" * 90)
print("     COMMENT EXAMPLES AT MIN, NEAR-MEAN, AND MAX CONFIDENCE PER EPISODE      ")
print("=" * 90)

val_labels = {1: "+1 (Favor)", 0: "0 (Neutral)", -1: "-1 (Against)"}

out_df = samples_df[[
    "Episode #", "Confidence_Tier", "confidence", "valence", "method", "text_snippet"
]].copy()

out_df.columns = ["Episode #", "Conf Tier", "Confidence", "Valence", "Method", "Comment Text Snippet"]
out_df["Confidence"] = out_df["Confidence"].map("{:.4f}".format)
out_df["Valence"] = out_df["Valence"].map(lambda v: val_labels.get(v, str(v)))

display(out_df)

     COMMENT EXAMPLES AT MIN, NEAR-MEAN, AND MAX CONFIDENCE PER EPISODE      


,Episode #,Conf Tier,Confidence,Valence,Method,Comment Text Snippet
0,1,Min Confidence,0.0000,0 (Neutral),VADER (Document),"Would it make sense to group Literals, Enums a..."
1,1,Near-Mean (0.672),0.7901,+1 (Favor),VADER (Document),"Jelle Zijlstra: In my reorganization PR, I add..."
2,1,Max Confidence,0.9923,+1 (Favor),VADER (Document),Now that the Typing Council ( PEP 729 ) is in ...
3,2,Min Confidence,0.0000,0 (Neutral),VADER (Document),I would propose that the spec should state tha...
4,2,Near-Mean (0.404),0.3400,-1 (Against),VADER (Document),The [typing spec](https://typing.readthedocs.i...
5,2,Max Confidence,0.6710,+1 (Favor),VADER (Document),Thanks @AlexWaygood and @JelleZijlstra! \r \r ...
6,9,Min Confidence,0.2449,-1 (Against),VADER (Document),I’d prefer to keep this out of the spec. We’ve...
7,9,Near-Mean (0.729),0.6908,+1 (Favor),VADER (Document),Thanks to everyone who provided feedback and P...
8,9,Max Confidence,0.9515,+1 (Favor),VADER (Document),I think my personal preference definitely is a...
9,13,Min Confidence,0.0000,0 (Neutral),VADER (Document),Eric Traut: Note that Annotated and Annotated[...
